In [2]:
import os, glob, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("Keras:", keras.__version__)


TF: 2.20.0
Keras: 3.13.2


In [3]:
PAMAP_ROOT = r"E:\WISDM_ar_v1.1\PAMAP2_Dataset"

RUN_NAME = "PAMAP2_TRANSFORMER_STREAMLINED"
OUT_DIR = os.path.join(PAMAP_ROOT, "OUTPUT_PAMAP2_TRANSFORMER", RUN_NAME)

DIRS = {
    "conf_mats": os.path.join(OUT_DIR, "confusion_matrices"),
    "models":    os.path.join(OUT_DIR, "models"),
    "plots":     os.path.join(OUT_DIR, "plots"),
    "reports":   os.path.join(OUT_DIR, "reports"),
    "tflite":    os.path.join(OUT_DIR, "tflite"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("OUT_DIR:", OUT_DIR)


OUT_DIR: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED


In [4]:
dat_files = sorted(glob.glob(os.path.join(PAMAP_ROOT, "**", "*.dat"), recursive=True))
print("Found .dat files:", len(dat_files))
for f in dat_files[:5]:
    print(" -", f)

Found .dat files: 28
 - E:\WISDM_ar_v1.1\PAMAP2_Dataset\Optional\subject101.dat
 - E:\WISDM_ar_v1.1\PAMAP2_Dataset\Optional\subject105.dat
 - E:\WISDM_ar_v1.1\PAMAP2_Dataset\Optional\subject106.dat
 - E:\WISDM_ar_v1.1\PAMAP2_Dataset\Optional\subject108.dat
 - E:\WISDM_ar_v1.1\PAMAP2_Dataset\Optional\subject109.dat


In [5]:
FEATURE_IDXS = [2,4,5,6,10,11,12,21,22,23,27,28,29,38,39,40,44,45,46]
FEATURE_NAMES = [
 "hr",
 "hand_acc_x","hand_acc_y","hand_acc_z",
 "hand_gyro_x","hand_gyro_y","hand_gyro_z",
 "chest_acc_x","chest_acc_y","chest_acc_z",
 "chest_gyro_x","chest_gyro_y","chest_gyro_z",
 "ankle_acc_x","ankle_acc_y","ankle_acc_z",
 "ankle_gyro_x","ankle_gyro_y","ankle_gyro_z"
]


In [6]:
def load_pamap2(files):
    dfs = []
    for f in files:
        df = pd.read_csv(f, sep=r"\s+", header=None)
        keep = [1] + FEATURE_IDXS  # col 1 = activity_id
        sub = df.iloc[:, keep].copy()
        sub.columns = ["activity_id"] + FEATURE_NAMES
        dfs.append(sub)
    return pd.concat(dfs, ignore_index=True)

data = load_pamap2(dat_files)
print("Raw shape:", data.shape)

# Remove activity 0 (unknown)
data = data[data.activity_id != 0]

# Clean NaN/Inf
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

activity_ids = sorted(data.activity_id.unique())
print("Clean shape:", data.shape)
print("Activities:", activity_ids)


Raw shape: (7701010, 20)
Clean shape: (493424, 20)
Activities: [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]


In [7]:
WIN  = 200
STEP = 100

id2idx = {a:i for i,a in enumerate(activity_ids)}
idx2id = {i:a for a,i in id2idx.items()}
NUM_CLASSES = len(activity_ids)

X = data[FEATURE_NAMES].values.astype(np.float32)
y = data.activity_id.map(id2idx).values.astype(np.int32)

def make_windows(X, y, win=WIN, step=STEP):
    Xw, yw = [], []
    for i in range(0, len(X) - win + 1, step):
        xs = X[i:i+win]
        ys = y[i:i+win]
        lab = np.bincount(ys).argmax()
        Xw.append(xs)
        yw.append(lab)
    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.int32)

Xw, yw = make_windows(X, y)
print("Windows:", Xw.shape, "Labels:", yw.shape)


Windows: (4933, 200, 19) Labels: (4933,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=SEED, stratify=yw
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_s  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

np.save(os.path.join(OUT_DIR, "X_train_s.npy"), X_train_s)
np.save(os.path.join(OUT_DIR, "X_test_s.npy"),  X_test_s)
np.save(os.path.join(OUT_DIR, "y_train.npy"),   y_train)
np.save(os.path.join(OUT_DIR, "y_test.npy"),    y_test)

with open(os.path.join(OUT_DIR, "labels.json"), "w") as f:
    json.dump({
        "activity_ids": [int(a) for a in activity_ids],
        "idx2id": {str(k): int(v) for k,v in idx2id.items()}
    }, f, indent=2)

print("✅ Saved arrays + labels.json")


✅ Saved arrays + labels.json


In [9]:
from tensorflow import keras
from tensorflow.keras import layers
from keras import ops  # ✅ Keras 3 safe ops (instead of tf.*)

def encoder(x, d_model, num_heads, mlp_dim, dropout):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ff = layers.Dense(mlp_dim, activation="gelu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    ff = layers.Dropout(dropout)(ff)

    return layers.LayerNormalization(epsilon=1e-6)(x + ff)

def build_transformer(input_shape, num_classes,
                      patch_len=10, d_model=96, depth=3,
                      num_heads=4, mlp_dim=192,
                      dropout=0.15, lr=1e-3):

    inp = keras.Input(shape=input_shape)  # (WIN, n_features)

    # Patch embedding
    x = layers.Conv1D(
        filters=d_model,
        kernel_size=patch_len,
        strides=patch_len,
        padding="valid"
    )(inp)

    # ✅ Keras 3 safe positional embedding using keras.ops
    # seq_len = ops.shape(x)[1]  -> dynamic length (symbolic OK)
    seq_len = ops.shape(x)[1]
    pos = ops.arange(0, seq_len, dtype="int32")              # (seq_len,)
    pos_emb = layers.Embedding(input_dim=2048, output_dim=d_model)(pos)  # (seq_len, d_model)
    pos_emb = ops.expand_dims(pos_emb, axis=0)               # (1, seq_len, d_model)
    x = x + pos_emb                                         # broadcast over batch

    x = layers.Dropout(dropout)(x)

    # Encoder stack
    for _ in range(depth):
        x = encoder(x, d_model, num_heads, mlp_dim, dropout)

    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---- Fixed hyperparameters (edit here only) ----
HP = dict(
    patch_len=10,
    d_model=96,
    depth=3,
    num_heads=4,
    mlp_dim=192,
    dropout=0.15,
    lr=1e-3,
)

best_model = build_transformer(
    input_shape=(WIN, X_train_s.shape[-1]),
    num_classes=NUM_CLASSES,
    **HP
)

best_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200, 19)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 20, 96)    │     18,336 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 96)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 20, 96)    │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 96)    │     37,248 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 96)    │          0 │ dropout[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 20, 96)    │        192 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20, 192)   │     18,624 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 20, 192)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 20, 96)    │     18,528 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 20, 96)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 20, 96)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 96)    │        192 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 96)    │     37,248 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 20, 96)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 96)    │        192 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 20, 192)   │     18,624 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 20, 192)   │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 20, 96)    │     18,528 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 20, 96)    │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 244,434 (954.82 KB)

 Trainable params: 244,434 (954.82 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5),
]

history = best_model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

keras_path = os.path.join(DIRS["models"], "transformer_best_fp32.keras")
best_model.save(keras_path)
print("✅ Saved:", keras_path)

Epoch 1/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 69ms/step - accuracy: 0.7165 - loss: 0.9697 - val_accuracy: 0.8818 - val_loss: 0.4118 - learning_rate: 0.0010
Epoch 2/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.8948 - loss: 0.3678 - val_accuracy: 0.9291 - val_loss: 0.2477 - learning_rate: 0.0010
Epoch 3/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9413 - loss: 0.2084 - val_accuracy: 0.9375 - val_loss: 0.1874 - learning_rate: 0.0010
Epoch 4/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9630 - loss: 0.1338 - val_accuracy: 0.9527 - val_loss: 0.1656 - learning_rate: 0.0010
Epoch 5/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9761 - loss: 0.0968 - val_accuracy: 0.9595 - val_loss: 0.1292 - learning_rate: 0.0010
Epoch 6/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.9809 - loss: 0.0759 - val_accuracy: 0.9459 - val_loss: 0.1615 - learning_rate: 0.0010
Epoch 7/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9878 - loss: 0.0572 - val_ac

In [11]:
CLASS_NAMES = [f"act_{a}" for a in activity_ids]

def plot_cm_values(cm, class_names, title, save_path, normalize=True):
    cm_plot = cm.astype(np.float32)
    if normalize:
        cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1e-9)

    plt.figure(figsize=(10, 9))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title, fontsize=14)
    plt.colorbar(fraction=0.046, pad=0.04)

    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right", fontsize=9)
    plt.yticks(ticks, class_names, fontsize=9)

    fmt = ".2f" if normalize else "d"
    thresh = cm_plot.max() * 0.55

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            v = cm_plot[i, j]
            plt.text(j, i, format(v, fmt),
                     ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if v > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()

probs = best_model.predict(X_test_s, batch_size=128, verbose=0)
y_pred = np.argmax(probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
cm_path = os.path.join(DIRS["conf_mats"], "TF_TRANSFORMER_FP32_cm_norm.png")
plot_cm_values(cm, CLASS_NAMES, "TF Transformer FP32 (Normalized CM)", cm_path, normalize=True)

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)
with open(os.path.join(DIRS["reports"], "TF_TRANSFORMER_FP32_report.txt"), "w") as f:
    f.write(report)

print("✅ Saved CM + report")
print(report[:800])


✅ Saved CM + report
              precision    recall  f1-score   support

       act_1       1.00      1.00      1.00        70
       act_2       0.97      0.99      0.98        67
       act_3       0.97      0.99      0.98        70
       act_4       1.00      1.00      1.00        85
       act_5       1.00      0.97      0.99        35
       act_6       0.98      0.97      0.97        60
       act_7       0.99      1.00      0.99        67
       act_9       1.00      1.00      1.00        31
      act_10       0.99      1.00      1.00       113
      act_11       1.00      1.00      1.00        20
      act_12       0.93      0.98      0.96        44
      act_13       1.00      0.94      0.97        36
      act_16       1.00      1.00      1.00        64
      act_17       0.97      0.99      0.98


In [12]:
_ = best_model.predict(X_test_s[:1], verbose=0)

SAVEDMODEL_DIR = os.path.join(DIRS["models"], "transformer_savedmodel")
if os.path.exists(SAVEDMODEL_DIR):
    shutil.rmtree(SAVEDMODEL_DIR)

best_model.export(SAVEDMODEL_DIR)
print("✅ Exported SavedModel:", SAVEDMODEL_DIR)

INFO:tensorflow:Assets written to: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\models\transformer_savedmodel\assets


INFO:tensorflow:Assets written to: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\models\transformer_savedmodel\assets


Saved artifact at 'E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\models\transformer_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 19), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 18), dtype=tf.float32, name=None)
Captures:
  3051694673744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694675664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694676432: TensorSpec(shape=(1, 20, 96), dtype=tf.float32, name=None)
  3051694677968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694677584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694678544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694677008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694678928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3051694678736: TensorSpec(shape=(), dtype=tf.resource, name=Non

In [13]:
def file_size_kb(path):
    return os.path.getsize(path) / 1024

TFLITE_PATHS = {
    "FP32":         os.path.join(DIRS["tflite"], "transformer_fp32.tflite"),
    "FP16":         os.path.join(DIRS["tflite"], "transformer_fp16.tflite"),
    "INT8_DYNAMIC": os.path.join(DIRS["tflite"], "transformer_int8_dynamic.tflite"),
    "INT8_FULL":    os.path.join(DIRS["tflite"], "transformer_int8_full.tflite"),
}

def representative_data():
    idx = np.random.choice(len(X_train_s), size=min(300, len(X_train_s)), replace=False)
    for i in idx:
        yield [X_train_s[i:i+1].astype(np.float32)]

def convert_fp32(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    open(out_path, "wb").write(c.convert())

def convert_fp16(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]
    open(out_path, "wb").write(c.convert())

def convert_int8_dynamic(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    open(out_path, "wb").write(c.convert())

def convert_int8_full(savedmodel_dir, out_path):
    c = tf.lite.TFLiteConverter.from_saved_model(savedmodel_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = representative_data
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.int8
    c.inference_output_type = tf.int8
    try:
        open(out_path, "wb").write(c.convert())
        return True
    except Exception as e:
        print("❌ INT8_FULL conversion failed:", str(e)[:350])
        return False

convert_fp32(SAVEDMODEL_DIR, TFLITE_PATHS["FP32"])
convert_fp16(SAVEDMODEL_DIR, TFLITE_PATHS["FP16"])
convert_int8_dynamic(SAVEDMODEL_DIR, TFLITE_PATHS["INT8_DYNAMIC"])
ok_full = convert_int8_full(SAVEDMODEL_DIR, TFLITE_PATHS["INT8_FULL"])
if not ok_full:
    TFLITE_PATHS.pop("INT8_FULL", None)

for k,p in TFLITE_PATHS.items():
    print(k, "->", file_size_kb(p), "KB", "|", p)


FP32 -> 995.703125 KB | E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\tflite\transformer_fp32.tflite
FP16 -> 520.5390625 KB | E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\tflite\transformer_fp16.tflite
INT8_DYNAMIC -> 314.265625 KB | E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\tflite\transformer_int8_dynamic.tflite
INT8_FULL -> 325.0625 KB | E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\tflite\transformer_int8_full.tflite


In [14]:
def tflite_predict(tflite_path, X, warmup=40, runs=250):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    in_idx = inp["index"]
    out_idx = out["index"]

    in_dtype = inp["dtype"]
    out_dtype = out["dtype"]

    in_scale, in_zero = inp.get("quantization", (1.0, 0))
    out_scale, out_zero = out.get("quantization", (1.0, 0))

    n = len(X)
    warmup = min(warmup, n)
    timed_runs = min(runs, n)

    probs = np.zeros((n, NUM_CLASSES), dtype=np.float32)

    def quantize_input(x_float):
        q = np.round(x_float / in_scale + in_zero)
        q = np.clip(q, -128, 127).astype(np.int8)
        return q

    def dequantize_output(y):
        return (y.astype(np.float32) - out_zero) * out_scale

    # Warmup
    for i in range(warmup):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)
        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()

    # Timed predictions
    t0 = time.perf_counter()
    for i in range(timed_runs):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)
        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()
        y = interpreter.get_tensor(out_idx)
        if out_dtype == np.int8:
            y = dequantize_output(y)
        probs[i] = y[0]
    latency_ms = (time.perf_counter() - t0) / max(timed_runs, 1) * 1000.0

    # Remaining predictions
    for i in range(timed_runs, n):
        x = X[i:i+1]
        if in_dtype == np.int8:
            x = quantize_input(x)
        interpreter.set_tensor(in_idx, x)
        interpreter.invoke()
        y = interpreter.get_tensor(out_idx)
        if out_dtype == np.int8:
            y = dequantize_output(y)
        probs[i] = y[0]

    return probs, latency_ms


In [15]:
def compute_metrics(y_true, probs):
    y_pred = np.argmax(probs, axis=1)

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

    mcc   = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    return y_pred, dict(
        accuracy=float(acc),
        balanced_accuracy=float(bacc),
        precision_macro=float(p_m),
        recall_macro=float(r_m),
        f1_macro=float(f1_m),
        precision_weighted=float(p_w),
        recall_weighted=float(r_w),
        f1_weighted=float(f1_w),
        mcc=float(mcc),
        kappa=float(kappa),
    )

def keras_latency_ms(model, X, warmup=50, runs=200):
    n = len(X)
    warmup = min(warmup, n)
    runs = min(runs, n)

    for i in range(warmup):
        _ = model.predict(X[i:i+1], verbose=0)
    t0 = time.perf_counter()
    for i in range(runs):
        _ = model.predict(X[i:i+1], verbose=0)
    return (time.perf_counter() - t0) / max(runs, 1) * 1000.0

rows = []

# ---- Keras FP32 baseline ----
probs_base = best_model.predict(X_test_s, batch_size=128, verbose=0)
y_pred_base, met_base = compute_metrics(y_test, probs_base)

keras_size_kb = os.path.getsize(keras_path) / 1024
lat_keras = keras_latency_ms(best_model, X_test_s)

cm = confusion_matrix(y_test, y_pred_base)
plot_cm_values(cm, CLASS_NAMES, "TF Transformer FP32 (Normalized CM)",
               os.path.join(DIRS["conf_mats"], "TF_TRANSFORMER_FP32_cm_norm.png"),
               normalize=True)

rows.append({
    "model": "TF_TRANSFORMER_FP32",
    "size_kb": float(keras_size_kb),
    "latency_ms": float(lat_keras),
    **met_base
})

# ---- TFLite variants ----
for name, path in TFLITE_PATHS.items():
    probs, lat = tflite_predict(path, X_test_s)
    y_pred, met = compute_metrics(y_test, probs)

    cm = confusion_matrix(y_test, y_pred)
    plot_cm_values(cm, CLASS_NAMES, f"TFLITE {name} (Normalized CM)",
                   os.path.join(DIRS["conf_mats"], f"TFLITE_TRANSFORMER_{name}_cm_norm.png"),
                   normalize=True)

    rows.append({
        "model": f"TFLITE_TRANSFORMER_{name}",
        "size_kb": float(os.path.getsize(path) / 1024),
        "latency_ms": float(lat),
        **met
    })

final_df = pd.DataFrame(rows)
final_csv = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
final_df.to_csv(final_csv, index=False)

print("✅ Saved final table:", final_csv)
print(final_df.sort_values("f1_macro", ascending=False))


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.

✅ Saved final table: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\summary_ALL_transformer_variants.csv
                             model      size_kb  latency_ms  accuracy  \
4     TFLITE_TRANSFORMER_INT8_FULL   325.062500    2.683156  0.984802   
0              TF_TRANSFORMER_FP32  3081.045898  141.267405  0.982776   
1          TFLITE_TRANSFORMER_FP32   995.703125    0.380794  0.982776   
2          TFLITE_TRANSFORMER_FP16   520.539062    0.375052  0.982776   
3  TFLITE_TRANSFORMER_INT8_DYNAMIC   314.265625    0.313255  0.982776   

   balanced_accuracy  precision_macro  recall_macro  f1_macro  \
4           0.979529         0.984674      0.979529  0.981938   
0           0.977786         0.980814      0.977786  0.979160   
1           0.977786         0.980814      0.977786  0.979160   
2           0.977786         0.980814      0.977786  0.979160   
3           0.977786         0.980814      0.977786  0.979160   

   precision_weighted  

In [16]:
df = pd.read_csv(os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv"))

keep = [
    "TFLITE_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC",
]
if "TFLITE_TRANSFORMER_INT8_FULL" in df["model"].values:
    keep.append("TFLITE_TRANSFORMER_INT8_FULL")

df = df[df["model"].isin(keep)].copy()
df["model"] = pd.Categorical(df["model"], categories=keep, ordered=True)
df = df.sort_values("model")

pretty = {
    "TFLITE_TRANSFORMER_FP32": "Transformer (FP32)",
    "TFLITE_TRANSFORMER_FP16": "Transformer (FP16)",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC": "Transformer (INT8 Dynamic)",
    "TFLITE_TRANSFORMER_INT8_FULL": "Transformer (INT8 Full)",
}
df["label"] = df["model"].map(pretty)

def hbar(y_labels, values, title, xlabel, save_path, fmt="{:.2f}"):
    y = np.arange(len(y_labels))
    plt.figure(figsize=(10, 4.5))
    bars = plt.barh(y, values)
    plt.yticks(y, y_labels, fontsize=11)
    plt.xlabel(xlabel)
    plt.title(title)
    vmax = np.nanmax(values)
    pad = vmax * 0.01
    for b,v in zip(bars, values):
        plt.text(v + pad, b.get_y()+b.get_height()/2, fmt.format(v), va="center")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()
    print("✅ Saved:", save_path)

hbar(df["label"], df["size_kb"].values,
     "Transformer deployment variants: model size", "Size (KB)",
     os.path.join(DIRS["plots"], "transformer_size_kb.png"), fmt="{:.0f}")

hbar(df["label"], (df["accuracy"].values * 100),
     "Transformer deployment variants: accuracy", "Accuracy (%)",
     os.path.join(DIRS["plots"], "transformer_accuracy.png"), fmt="{:.2f}")

hbar(df["label"], (df["f1_macro"].values * 100),
     "Transformer deployment variants: F1-macro", "F1-macro (%)",
     os.path.join(DIRS["plots"], "transformer_f1_macro.png"), fmt="{:.2f}")


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\transformer_size_kb.png
✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\transformer_accuracy.png
✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\transformer_f1_macro.png


In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc


In [18]:
def micro_macro_roc(y_true, probs, n_classes):
    """
    Multiclass ROC using One-vs-Rest, returning:
    (fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro)
    """
    y_true = np.asarray(y_true).astype(int)
    probs  = np.asarray(probs).astype(np.float32)

    y_bin = label_binarize(y_true, classes=np.arange(n_classes))

    # Micro-average
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), probs.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)

    # Macro-average
    fpr = {}
    tpr = {}
    for c in range(n_classes):
        fpr[c], tpr[c], _ = roc_curve(y_bin[:, c], probs[:, c])

    all_fpr = np.unique(np.concatenate([fpr[c] for c in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)

    for c in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])

    mean_tpr /= n_classes
    fpr_macro, tpr_macro = all_fpr, mean_tpr
    auc_macro = auc(fpr_macro, tpr_macro)

    return fpr_micro, tpr_micro, auc_micro, fpr_macro, tpr_macro, auc_macro


In [19]:
# -------- Collect curves (TF + all TFLite variants) --------
n_classes = NUM_CLASSES

curves = []

# TF FP32 baseline (Keras model)
probs_tf = best_model.predict(X_test_s, batch_size=256, verbose=0)
fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tf, n_classes)
curves.append({
    "name": "TF FP32",
    "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
    "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
})

# TFLite variants (LiteRT inference via your tflite_predict)
for key, path in TFLITE_PATHS.items():
    probs_tfl, _ = tflite_predict(path, X_test_s)
    fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tfl, n_classes)
    curves.append({
        "name": f"TFLite {key}",
        "fpr_micro": fmi, "tpr_micro": tmi, "auc_micro": aucmi,
        "fpr_macro": fma, "tpr_macro": tma, "auc_macro": aucma,
    })

# -------- Two-panel figure: Micro (left) + Macro (right) --------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.6))

# Left: Micro
for c in curves:
    ax1.plot(c["fpr_micro"], c["tpr_micro"], linewidth=2.0,
             label=f"{c['name']} (AUC={c['auc_micro']:.4f})")
ax1.plot([0, 1], [0, 1], linestyle=":", linewidth=1.5)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("Micro-average ROC (OvR)")
ax1.grid(alpha=0.25)

# Right: Macro
for c in curves:
    ax2.plot(c["fpr_macro"], c["tpr_macro"], linewidth=2.0,
             label=f"{c['name']} (AUC={c['auc_macro']:.4f})")
ax2.plot([0, 1], [0, 1], linestyle=":", linewidth=1.5)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("Macro-average ROC (OvR)")
ax2.grid(alpha=0.25)

# Shared legend outside
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)

fig.suptitle("PAMAP2 Transformer: TF FP32 vs TFLite Variants (ROC)", fontsize=13)
plt.tight_layout(rect=[0, 0, 0.82, 0.95])

out_path = os.path.join(DIRS["plots"], "PAMAP2_ROC_TWOPANEL_micro_macro.png")
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

print("✅ Saved two-panel ROC:", out_path)

# -------- Save AUC summary (micro+macro) --------
summary = pd.DataFrame([{
    "model": c["name"],
    "auc_micro": float(c["auc_micro"]),
    "auc_macro": float(c["auc_macro"]),
} for c in curves]).sort_values("auc_macro", ascending=False)

summary_path = os.path.join(OUT_DIR, "PAMAP2_ROC_AUC_micro_macro_summary.csv")
summary.to_csv(summary_path, index=False)

print("✅ Saved AUC summary:", summary_path)
print(summary)


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.

✅ Saved two-panel ROC: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_ROC_TWOPANEL_micro_macro.png
✅ Saved AUC summary: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\PAMAP2_ROC_AUC_micro_macro_summary.csv
                 model  auc_micro  auc_macro
0              TF FP32   0.999690   0.999703
1          TFLite FP32   0.999690   0.999703
2          TFLite FP16   0.999690   0.999703
3  TFLite INT8_DYNAMIC   0.999690   0.999701
4     TFLite INT8_FULL   0.997868   0.996239


In [20]:
def plot_micro_macro_only(fpr_micro, tpr_micro, auc_micro,
                          fpr_macro, tpr_macro, auc_macro,
                          title, save_path):
    plt.figure(figsize=(8.8, 6.6))
    plt.plot(fpr_micro, tpr_micro, linewidth=2.2, label=f"micro (AUC={auc_micro:.4f})")
    plt.plot(fpr_macro, tpr_macro, linewidth=2.2, linestyle="--", label=f"macro (AUC={auc_macro:.4f})")
    plt.plot([0, 1], [0, 1], linestyle=":", linewidth=1.4)
    plt.xlim([0, 1])
    plt.ylim([0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()
    print("✅ Saved:", save_path)

# TF baseline
fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs_tf, NUM_CLASSES)
plot_micro_macro_only(
    fmi, tmi, aucmi, fma, tma, aucma,
    "PAMAP2 TF Transformer FP32 - ROC (micro & macro)",
    os.path.join(DIRS["plots"], "PAMAP2_TF_FP32_ROC_micro_macro.png")
)

# TFLite variants
for key, path in TFLITE_PATHS.items():
    probs, _ = tflite_predict(path, X_test_s)
    fmi, tmi, aucmi, fma, tma, aucma = micro_macro_roc(y_test, probs, NUM_CLASSES)
    plot_micro_macro_only(
        fmi, tmi, aucmi, fma, tma, aucma,
        f"PAMAP2 TFLite {key} - ROC (micro & macro)",
        os.path.join(DIRS["plots"], f"PAMAP2_TFLITE_{key}_ROC_micro_macro.png")
    )


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_TF_FP32_ROC_micro_macro.png


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_TFLITE_FP32_ROC_micro_macro.png


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_TFLITE_FP16_ROC_micro_macro.png


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_TFLITE_INT8_DYNAMIC_ROC_micro_macro.png


e:\WISDM_ar_v1.1\aasif_env\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ Saved: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_TFLITE_INT8_FULL_ROC_micro_macro.png


In [21]:
main_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
if not os.path.exists(main_path):
    raise FileNotFoundError("Missing main summary file: " + main_path)

df_main = pd.read_csv(main_path)
df_auc  = pd.read_csv(os.path.join(OUT_DIR, "PAMAP2_ROC_AUC_micro_macro_summary.csv"))

# Make model names align
# df_auc has names like "TF FP32", "TFLite FP16" etc.
# df_main has names like "TF_TRANSFORMER_FP32", "TFLITE_TRANSFORMER_FP16" etc.
name_map = {"TF FP32": "TF_TRANSFORMER_FP32"}
for k in TFLITE_PATHS.keys():
    name_map[f"TFLite {k}"] = f"TFLITE_TRANSFORMER_{k}"

df_auc["model_key"] = df_auc["model"].map(name_map)
df_auc = df_auc.dropna(subset=["model_key"])[["model_key","auc_micro","auc_macro"]]

df_out = df_main.merge(df_auc, left_on="model", right_on="model_key", how="left").drop(columns=["model_key"])

out_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants_with_ROC.csv")
df_out.to_csv(out_path, index=False)

print("✅ Saved merged summary with ROC AUC:", out_path)
print(df_out)


✅ Saved merged summary with ROC AUC: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\summary_ALL_transformer_variants_with_ROC.csv
                             model      size_kb  latency_ms  accuracy  \
0              TF_TRANSFORMER_FP32  3081.045898  141.267405  0.982776   
1          TFLITE_TRANSFORMER_FP32   995.703125    0.380794  0.982776   
2          TFLITE_TRANSFORMER_FP16   520.539062    0.375052  0.982776   
3  TFLITE_TRANSFORMER_INT8_DYNAMIC   314.265625    0.313255  0.982776   
4     TFLITE_TRANSFORMER_INT8_FULL   325.062500    2.683156  0.984802   

   balanced_accuracy  precision_macro  recall_macro  f1_macro  \
0           0.977786         0.980814      0.977786  0.979160   
1           0.977786         0.980814      0.977786  0.979160   
2           0.977786         0.980814      0.977786  0.979160   
3           0.977786         0.980814      0.977786  0.979160   
4           0.979529         0.984674      0.979529  0.981938   

In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Load summary ----
summary_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
df = pd.read_csv(summary_path)

# ✅ Force to plain string (prevents categorical category mismatch errors)
df["model"] = df["model"].astype(str)

# ---- Keep/order models ----
keep_order = [
    "TF_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP32",
    "TFLITE_TRANSFORMER_FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC",
    "TFLITE_TRANSFORMER_INT8_FULL",
]
keep_order = [m for m in keep_order if m in df["model"].values]  # only those present

df = df[df["model"].isin(keep_order)].copy()

# ✅ Safe categorical assignment (now model is string)
df["model"] = pd.Categorical(df["model"], categories=keep_order, ordered=True)
df = df.sort_values("model")

# Pretty names
pretty = {
    "TF_TRANSFORMER_FP32": "TF FP32",
    "TFLITE_TRANSFORMER_FP32": "TFLite FP32",
    "TFLITE_TRANSFORMER_FP16": "TFLite FP16",
    "TFLITE_TRANSFORMER_INT8_DYNAMIC": "TFLite INT8-Dyn",
    "TFLITE_TRANSFORMER_INT8_FULL": "TFLite INT8-Full",
}
df["label"] = df["model"].astype(str).map(pretty).fillna(df["model"].astype(str))

# Ensure numeric
for c in ["accuracy", "f1_macro", "size_kb", "latency_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- Normalize metrics for a single-axis plot ----
def minmax_norm(arr):
    arr = np.asarray(arr, dtype=np.float64)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

acc_n  = minmax_norm(df["accuracy"].values)
f1_n   = minmax_norm(df["f1_macro"].values)
size_n = minmax_norm(df["size_kb"].values)
lat_n  = minmax_norm(df["latency_ms"].values)

labels = df["label"].tolist()
x = np.arange(len(labels))
bar_w = 0.20

fig, ax = plt.subplots(figsize=(13.5, 5.8))

b1 = ax.bar(x - 1.5*bar_w, acc_n,  width=bar_w, label="Accuracy (norm)")
b2 = ax.bar(x - 0.5*bar_w, f1_n,   width=bar_w, label="F1-macro (norm)")
b3 = ax.bar(x + 0.5*bar_w, size_n, width=bar_w, label="Model size KB (norm)")
b4 = ax.bar(x + 1.5*bar_w, lat_n,  width=bar_w, label="Latency ms (norm)")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Normalized value (0–1)", fontsize=12)
ax.set_title("PAMAP2 Transformer Variants: Accuracy, F1, Model Size, Latency (single view)", fontsize=13)
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper left", fontsize=10)

# ---- Annotate bars with REAL values ----
def annotate(bars, real_vals, fmt):
    for bar, v in zip(bars, real_vals):
        if not np.isfinite(v):
            continue
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            fmt.format(v),
            ha="center", va="bottom",
            fontsize=9, rotation=90
        )

annotate(b1, df["accuracy"].values * 100.0, "{:.2f}%")
annotate(b2, df["f1_macro"].values * 100.0, "{:.2f}%")
annotate(b3, df["size_kb"].values, "{:.0f}KB")
annotate(b4, df["latency_ms"].values, "{:.2f}ms")

plt.tight_layout()
out_path = os.path.join(DIRS["plots"], "PAMAP2_single_bar_acc_f1_size_latency.png")
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

print("✅ Saved single bar graph:", out_path)
print(df[["model","accuracy","f1_macro","size_kb","latency_ms"]])


✅ Saved single bar graph: E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\plots\PAMAP2_single_bar_acc_f1_size_latency.png
                             model  accuracy  f1_macro      size_kb  \
0              TF_TRANSFORMER_FP32  0.982776  0.979160  3081.045898   
1          TFLITE_TRANSFORMER_FP32  0.982776  0.979160   995.703125   
2          TFLITE_TRANSFORMER_FP16  0.982776  0.979160   520.539062   
3  TFLITE_TRANSFORMER_INT8_DYNAMIC  0.982776  0.979160   314.265625   
4     TFLITE_TRANSFORMER_INT8_FULL  0.984802  0.981938   325.062500   

   latency_ms  
0  141.267405  
1    0.380794  
2    0.375052  
3    0.313255  
4    2.683156  


In [23]:
import os
import pandas as pd
import numpy as np

# ---- Load your main summary ----
summary_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants.csv")
df = pd.read_csv(summary_path)

# Ensure numeric
for c in ["size_kb", "latency_ms"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# -------------------------
# 1) FPS (Frames per second)
# -------------------------
# latency_ms -> fps = 1000 / latency_ms
df["fps"] = 1000.0 / df["latency_ms"]

# Handle any division issues
df["fps"] = df["fps"].replace([np.inf, -np.inf], np.nan)

# -------------------------
# 2) Compression ratio
# -------------------------
# Reference FP32 Keras model size
# (TF_TRANSFORMER_FP32 row)

fp32_rows = df[df["model"] == "TF_TRANSFORMER_FP32"]

if len(fp32_rows) == 0:
    raise ValueError("❌ TF_TRANSFORMER_FP32 row not found in summary table!")

fp32_size = float(fp32_rows["size_kb"].values[0])

print("FP32 reference size (KB):", fp32_size)

# Compression ratio = FP32_size / model_size
df["compression_ratio"] = fp32_size / df["size_kb"]

# For FP32 baseline itself, set ratio = 1
df.loc[df["model"] == "TF_TRANSFORMER_FP32", "compression_ratio"] = 1.0

# -------------------------
# Save updated table
# -------------------------
out_path = os.path.join(OUT_DIR, "summary_ALL_transformer_variants_with_FPS_CR.csv")
df.to_csv(out_path, index=False)

print("✅ Saved updated summary with FPS + Compression Ratio:")
print(out_path)

# Show important columns nicely
display_cols = [
    "model",
    "accuracy",
    "f1_macro",
    "size_kb",
    "latency_ms",
    "fps",
    "compression_ratio"
]

print(df[display_cols].sort_values("compression_ratio", ascending=False))


FP32 reference size (KB): 3081.0458984375
✅ Saved updated summary with FPS + Compression Ratio:
E:\WISDM_ar_v1.1\PAMAP2_Dataset\OUTPUT_PAMAP2_TRANSFORMER\PAMAP2_TRANSFORMER_STREAMLINED\summary_ALL_transformer_variants_with_FPS_CR.csv
                             model  accuracy  f1_macro      size_kb  \
3  TFLITE_TRANSFORMER_INT8_DYNAMIC  0.982776  0.979160   314.265625   
4     TFLITE_TRANSFORMER_INT8_FULL  0.984802  0.981938   325.062500   
2          TFLITE_TRANSFORMER_FP16  0.982776  0.979160   520.539062   
1          TFLITE_TRANSFORMER_FP32  0.982776  0.979160   995.703125   
0              TF_TRANSFORMER_FP32  0.982776  0.979160  3081.045898   

   latency_ms          fps  compression_ratio  
3    0.313255  3192.285396           9.803955  
4    2.683156   372.695438           9.478318  
2    0.375052  2666.299784           5.918952  
1    0.380794  2626.089038           3.094342  
0  141.267405     7.078774           1.000000  
